In [17]:
import pandas as pd
import joblib
import os

In [18]:
xgb_model = joblib.load('../models/xgboost_model.pkl')
encoders = joblib.load('../models/feature_encoders.pkl')
y_encoder = joblib.load('../models/target_encoder.pkl')
Features = joblib.load('../models/feature_names.pkl')
preprocessing_info = joblib.load("../models/preprocessing_info.pkl")

print('Loaded model expecting features:', Features)
print("All saved files loaded successfully.")

Loaded model expecting features: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
All saved files loaded successfully.


In [19]:
test_df = pd.read_csv('../dataset/test.csv')
print(test_df.shape)
test_df.head()

(295753, 14)


,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [20]:
test_ids = test_df["id"]

test_df = test_df.drop("id", axis=1)

In [21]:
for col in preprocessing_info["numeric_columns"]:
    median = preprocessing_info["numeric_medians"][col]
    test_df[col] = test_df[col].fillna(median)

for col in preprocessing_info["categorical_columns"]:
    test_df[col] = test_df[col].fillna(
        preprocessing_info["categorical_fill"]
    )    

In [29]:
for col in preprocessing_info["categorical_columns"]:
    encoder = encoders[col]

    known_values = set(encoder.classes_)

    test_df[col] = test_df[col].apply(
        lambda x: x if x in known_values else "missing"
    )

    test_df[col] = encoder.transform(test_df[col])

In [23]:
test_df = test_df[Features]

In [24]:
predictions = xgb_model.predict(test_df)

print(predictions[:10])

[2 2 0 0 2 0 0 0 0 0]


In [25]:
predictions = y_encoder.inverse_transform(predictions)

print(predictions[:10])

['unhealthy' 'unhealthy' 'at-risk' 'at-risk' 'unhealthy' 'at-risk'
 'at-risk' 'at-risk' 'at-risk' 'at-risk']


In [27]:
submission = pd.DataFrame({

    "id": test_ids,

    "health_condition": predictions

})

submission.head()


submission.to_csv(

    "../Submissions/xgboost_submission.csv",

    index=False

)

print("submission.csv created successfully.")

submission.csv created successfully.


In [28]:
print(submission.head())

print()

print(submission.shape)

       id health_condition
0  690088        unhealthy
1  690089        unhealthy
2  690090          at-risk
3  690091          at-risk
4  690092        unhealthy

(295753, 2)
